# ***Retail Sales Analysis: Exploring Walmart Store Performance and Holiday Impact***

---



**Project Objective**

This stage of the Walmart sales analysis focuses on predictive modeling. Building on the exploratory, SQL, statistical, and Power BI analysis completed earlier in the project, this notebook examines how well the available store, holiday, economic, environmental, and seasonal variables can predict weekly sales.

A linear regression model is used as an interpretable baseline. Model performance is evaluated using Mean Absolute Error (MAE) and R², with particular attention to the limitations of the available features and the importance of correctly handling store and time-related information.



---



In [7]:
import pandas as pd
import sqlite3
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
#The following libraries are used for data analysis, SQL querying, statistical testing, and visualization.

In [2]:
from google.colab import files
uploaded = files.upload()

filename = next(iter(uploaded))

Saving walmart-sales-dataset.csv to walmart-sales-dataset.csv


In [5]:
df = pd.read_csv("walmart-sales-dataset.csv")

df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106



---
# **Data Quality and Cleaning:**


**Number of Rows and Columns**

In [ ]:
df.shape

**Checking For Missing Values, Duplicates and Data Type:**

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe()

The statistical summary provides an overview of the dataset, including the average, standard deviation, minimum, maximum, and quartile values for each numerical variable. These values help confirm that the dataset was loaded correctly and provide an initial understanding of the range and distribution of the data.

# Convert Date from text (DD-MM-YYYY) into a proper datetime datatype:

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y"
)
df.dtypes

# Date Range:

In [ ]:
print("Earliest Date:", df["Date"].min())
print("Latest Date:", df["Date"].max())

In [ ]:
df = df.sort_values(
    by=["Store", "Date"]
).reset_index(drop=True)
df.head(5)



---


**Calculating Total Sales:**

In [ ]:
total_sales = df['Weekly_Sales'].sum()
print("Total Sales:", total_sales)



---


# **Loading the Data into SQLite:**

In [ ]:
conn = sqlite3.connect("walmart.db")
df.to_sql("walmart_sales", conn, if_exists="replace", index=False)


In [ ]:
query = """
SELECT *
FROM walmart_sales
LIMIT 10;
"""
pd.read_sql_query(query, conn)

**Validating the SQLite:**

1. Counting rows

In [ ]:
pd.read_sql_query("""
SELECT COUNT(*) AS Row_Count
FROM walmart_sales;
""", conn)

2. Verifing the date range matches the original Dataset

In [ ]:
pd.read_sql_query("""
SELECT MIN(Date) AS Earliest_Date, MAX(Date) AS Latest_Date
FROM walmart_sales;
""", conn)

3. Number of Stores on the sql table:

In [ ]:
pd.read_sql_query("""
SELECT COUNT(DISTINCT Store) AS Number_of_Stores
FROM walmart_sales;
""", conn)

4. Verifing summary statistics against the original dataset

In [ ]:
pd.read_sql_query("""
SELECT MIN(Weekly_Sales) AS Min_Sales, MAX(Weekly_Sales) AS Max_Sales, AVG(Weekly_Sales) AS Avg_Sales, SUM(Weekly_Sales) AS Total_Sales
FROM walmart_sales;
""", conn)

5. Checking SQL column data types

In [ ]:
pd.read_sql_query("""
PRAGMA table_info(walmart_sales);
""", conn)



---


# **Overall Sales KPIs:**

In [ ]:
pd.read_sql_query("""
SELECT AVG(Weekly_Sales) AS Avg_weekly_sales, SUM(Weekly_Sales) AS Total_Sales
FROM walmart_sales;
""", conn)




**Total Sales by Each Store:**

This analysis calculates the total sales generated by each store and ranks the stores from highest to lowest based on overall sales performance.


In [ ]:
pd.read_sql_query("""
SELECT Store, SUM(Weekly_Sales) AS Total_Sales, RANK() OVER (ORDER BY SUM(Weekly_Sales) DESC) AS Sales_Rank
FROM walmart_sales
GROUP BY Store
ORDER BY Sales_Rank;
""",conn)

**The Average Weekly Sales by Each Store:**

This analysis calculates the average weekly sales for each of the 45 stores to compare typical sales performance across locations.

In [ ]:
pd.read_sql_query("""
SELECT Store, AVG(Weekly_Sales) AS Avg_Weekly_Sales
FROM walmart_sales
GROUP BY Store
Order by Avg_Weekly_Sales DESC

  """, conn)

# Store Performance Compared With the Overall Average



To better understand store-level performance, each store’s average weekly sales are compared with the overall average weekly sales across all store-week observations.

Average weekly sales of 45 stores is \$1.047 million

In [ ]:
pd.read_sql_query("""
SELECT Store, AVG(Weekly_Sales) AS Avg_Weekly_Sales_of_each_store, AVG(Weekly_Sales) - (
    SELECT AVG(Weekly_Sales) FROM walmart_sales) AS Difference_From_Overall_Avg,
    CASE
        WHEN AVG(Weekly_Sales) >= (
            SELECT AVG(Weekly_Sales)
            FROM walmart_sales
        )
        THEN 'Above Average'
        ELSE 'Below Average'
    END AS Performance
FROM walmart_sales
GROUP BY Store
ORDER BY Difference_From_Overall_Avg DESC;
""",conn)

A positive difference indicates that a store performs above the overall average, while a negative difference indicates that it performs below the overall average.



---

# **Time-Based Analysis:**

**Total Sales by Year:**


This analysis examines Walmart’s sales performance across the years included in the dataset. Because 2010 and 2012 contain fewer observed weeks than 2011, total annual sales should be interpreted alongside the number of weeks available for each year. Average weekly sales provide a more comparable measure of performance across years.

In [ ]:
pd.read_sql_query("""
SELECT
    strftime('%Y', Date) AS Year,
    COUNT(DISTINCT Date) AS Weeks_Observed,
    SUM(Weekly_Sales) AS Total_Sales,
    AVG(Weekly_Sales) AS Avg_Weekly_Sales
FROM walmart_sales
GROUP BY Year
ORDER BY Year;
""", conn)



---


**Holiday Weeks vs Regular Weeks:**


This analysis compares average weekly sales during holiday weeks with regular weeks to determine whether noticeable differences in sales performance exist between the two periods.


In [ ]:
pd.read_sql_query("""
Select
    CASE
       When Holiday_Flag = 1 THEN 'Holiday_week'
       Else 'Regular_week' END AS Holiday_Type,
       COUNT(DISTINCT DATE) AS Number_of_Weeks,
       COUNT(*) AS Store_week_Observations,
       SUM(Weekly_Sales) AS Total_Sales,
       AVG(Weekly_Sales) AS Avg_Weekly_Sales
FROM walmart_sales
GROUP BY Holiday_Flag;
""",conn)



The following chart compares average weekly sales during holiday and regular weeks. This provides a visual view of the difference identified in the statistical analysis and helps show the magnitude of the holiday sales increase.

In [ ]:
# Calculate average sales for holiday and regular weeks
holiday_comparison = (
    df.groupby("Holiday_Flag")["Weekly_Sales"]
      .mean()
)

holiday_labels = ["Regular Weeks", "Holiday Weeks"]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    holiday_labels,
    holiday_comparison.values
)

plt.title("Average Weekly Sales: Holiday vs. Regular Weeks")
plt.ylabel("Average Weekly Sales")

# Format y-axis as dollars
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"${x/1_000_000:.2f}M")
)

# Add sales values above bars
for bar in bars:
    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"${height/1_000_000:.2f}M",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

Average weekly sales per store are visibly higher during holiday weeks than during regular weeks. This visual pattern is consistent with the paired-samples t-test, which found a statistically significant difference between the two periods. While the results suggest that holiday periods are associated with stronger sales, other seasonal and business factors may also contribute to the increase.

---



**Which Stores Benefit Most During Holidays?**

This analysis compares each store’s average holiday-week sales with its average regular-week sales to identify which stores experience the largest increase during holiday periods.


In [ ]:
pd.read_sql_query("""
SELECT
    Store,
    AVG(CASE
        WHEN Holiday_Flag = 1 THEN Weekly_Sales
    END) AS Holiday_Avg_Sales,

    AVG(CASE
        WHEN Holiday_Flag = 0 THEN Weekly_Sales
    END) AS Regular_Avg_Sales,

    AVG(CASE
        WHEN Holiday_Flag = 1 THEN Weekly_Sales
    END)
    -
    AVG(CASE
        WHEN Holiday_Flag = 0 THEN Weekly_Sales
    END) AS Sales_Difference,
   ROUND(
    (
        AVG(CASE WHEN Holiday_Flag = 1 THEN Weekly_Sales END)
        -
        AVG(CASE WHEN Holiday_Flag = 0 THEN Weekly_Sales END)
    )
    /
    AVG(CASE WHEN Holiday_Flag = 0 THEN Weekly_Sales END)
    * 100,
    2
) AS Holiday_Lift_Percent

FROM walmart_sales
GROUP BY Store
ORDER BY Sales_Difference DESC;

""",conn)

Store 10 had the largest absolute holiday increase. However, store 7 has the largest percentage increase.

---



**Which Months Have the Highest Average Weekly Sales?**

This analysis compares average weekly sales by month to identify seasonal patterns and determine which months tend to generate stronger weekly sales performance.

In [ ]:
pd.read_sql_query("""
SELECT
    strftime('%m', Date) AS Month_Number,
    AVG(Weekly_Sales) AS Avg_Weekly_Sales,
    COUNT(*) AS Observations
FROM walmart_sales
GROUP BY Month_Number
ORDER BY Avg_Weekly_Sales DESC;

""", conn)

# **Holiday Sales: Statistical Significance**
Holiday weeks show higher average sales than regular weeks in the dataset. To determine whether this difference is statistically significant, a paired-samples t-test is used to compare each store’s average holiday-week sales with the same store’s average regular-week sales.



Average holiday-week sales are approximately \$1.12 million per store, compared with approximately \$1.04 million during regular weeks.


In [ ]:

store_holiday = (
    df.groupby(["Store", "Holiday_Flag"])["Weekly_Sales"]
      .mean()
      .unstack()
)

t_stat, p_value = stats.ttest_rel(
    store_holiday[1],
    store_holiday[0]
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

Statistical Test Result

The paired-samples t-test produced a t-statistic of approximately 9.65 and a p-value of approximately 1.97 × 10⁻¹², which is well below the 0.05 significance level.

This provides strong statistical evidence that average sales differ between holiday and regular weeks across the stores in the dataset. However, statistical significance alone does not establish that holidays directly caused the increase in sales. Other seasonal or business-related factors may also contribute to the difference.



---



# **Economic/Environmental Factors**


**Temperature, Fuel Price, CPI, and Unemployment**

This analysis examines whether temperature, fuel prices, CPI, and unemployment are associated with weekly sales. Pearson correlation coefficients are used to measure the strength and direction of the linear relationship between each factor and weekly sales.


In [ ]:
factors = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
factors_relationship = df[factors].corr()
factors_relationship["Weekly_Sales"].sort_values(ascending=False)

Correlation Results

The analysis produced the following correlations with weekly sales:

1. Fuel Price: very weak positive relationship, r = 0.009
2. Temperature: very weak negative relationship, r = -0.064
3. CPI: very weak negative relationship, r = -0.073
4. Unemployment: weak negative relationship, r = -0.106

All four correlation coefficients are close to zero, indicating weak linear relationships with weekly sales. Based on this analysis, none of these variables alone explains a large portion of the variation in sales.

However, these results do not mean that the factors have no effect on sales. They only indicate that none shows a strong standalone linear relationship with weekly sales in this dataset. Correlation also does not establish causation.



---

Sales Distribution Analysis

Distribution of Weekly Sales

This analysis examines the distribution of weekly sales to understand typical sales levels, variability, and the presence of unusually high or low observations.

The average weekly sales per store are approximately \$1.047 million, while the median is approximately \$961,000. Because the mean is higher than the median, larger sales observations are pulling the average upward.

The standard deviation is approximately \$564,000, indicating substantial variation in weekly sales across stores and time periods. The middle 50% of observations range from approximately \$553,000 to \$1.42 million, further demonstrating the wide variation in store-week performance.

The skewness value of 0.668 indicates that the distribution is moderately right-skewed. This means that while most observations are concentrated at lower and middle sales levels, several high-sales observations extend the upper end of the distribution.


In [ ]:
df["Weekly_Sales"].describe()

# Calculating the Key Numbers Separately:

In [ ]:
mean = df["Weekly_Sales"].mean()
median = df["Weekly_Sales"].median()
std = df["Weekly_Sales"].std()

q1 = df["Weekly_Sales"].quantile(0.25)
q3 = df["Weekly_Sales"].quantile(0.75)

iqr = q3 - q1

skew = df["Weekly_Sales"].skew()

print("Mean:", mean)
print("Median:", median)
print("Standard Deviation:", std)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Skewness:", skew)

**Distribution of Walmart Weekly Sales:**

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(df["Weekly_Sales"], bins=30)

plt.xlabel("Weekly Sales")
plt.ylabel("Frequency")
plt.title("Distribution of Walmart Weekly Sales")

plt.show()

Conclusion:

Mean:	\$1,046,964.88 (Average weekly sales)

Median	\$960,746.04	(Typical middle observation)

Std. deviation	\$564,366.62	(Large variation in weekly sales)

Q1: \$553,350.11
25% of store-week observations had sales below this value.

Q3: \$1,420,158.66
75% of store-week observations had sales below this value.

IQR	\$866,808.55	(Spread of the middle 50%)

Skewness	0.668	(Moderately right-skewed)

Minimum	\$209,986.25	(Lowest weekly sales)

Maximum	\$3,818,686.45	(Highest weekly sales)



---

# **Statistical and Business Analysis Conclusion**

The statistical and business analysis provides a clearer understanding of Walmart’s weekly sales performance across the 45 stores in the dataset.

Store performance varies considerably across locations, with 19 stores recording average weekly sales above the overall average and 26 stores falling below it. This suggests that company-wide averages alone may hide meaningful differences between individual store locations.

Holiday weeks also recorded higher average sales than regular weeks. A paired-samples t-test produced a t-statistic of approximately 9.65 and a p-value below 0.001, providing strong statistical evidence of a difference between holiday and regular-week sales. However, this relationship should not be interpreted as proof that holidays directly caused the increase.

The economic and environmental variables included in the dataset showed only weak linear relationships with weekly sales. Temperature, fuel prices, CPI, and unemployment all produced correlation coefficients close to zero, suggesting that these variables alone do not explain much of the variation in sales.

Weekly sales also showed substantial variability. Average weekly sales per store were approximately \$1.047 million, compared with a median of approximately \$961,000 and a standard deviation of approximately \$564,000. The positive skewness of 0.668 indicates that higher-sales observations pull the overall average upward.

Overall, the findings suggest that store-level differences, timing, and holiday periods are important areas to consider when evaluating Walmart’s weekly sales performance.



---



# **Business Insights**

The analysis identified several meaningful patterns in Walmart’s weekly sales performance.

First, sales performance varies significantly across individual stores. Of the 45 stores analyzed, 19 recorded average weekly sales above the overall average, while 26 fell below it. This shows that performance is not evenly distributed across locations and that company-wide averages can hide important store-level differences.

Holiday weeks also produced higher average sales than regular weeks. The paired statistical test found a significant difference between the two periods, suggesting that holiday periods are an important consideration when planning inventory, staffing, and other store resources. However, the analysis does not establish that holidays alone caused the increase in sales.

The sales distribution also shows substantial variation across store-week observations. The average weekly sales per store were approximately \$1.047 million, while the median was approximately \$961,000. This difference, along with a standard deviation of approximately \$564,000 and positive skewness of 0.668, indicates that some high-sales observations pull the average upward.

Finally, temperature, fuel prices, CPI, and unemployment showed only weak linear relationships with weekly sales. This suggests that these variables alone are not sufficient to explain Walmart’s sales performance and that additional factors such as store characteristics, promotions, local demand, product mix, and seasonal events may provide greater explanatory value.